In [38]:
from typing import TypedDict, Annotated, Literal
import sqlite3
import requests

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, SystemMessage,HumanMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_ollama import OllamaEmbeddings
load_dotenv()
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [13]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)
embeddings =OllamaEmbeddings(model="nomic-embed-text", base_url="http://172.31.0.1:11434")

In [14]:
loader = PyPDFLoader("Sanjay_Jat_Resume .pdf")
docs=loader.load()

In [15]:
len(docs)

1

In [16]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks=splitter.split_documents(docs)

In [17]:
len(chunks)

11

In [19]:
vector_store=FAISS.from_documents(chunks,embeddings)

In [21]:
vector_store

In [22]:
retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [24]:
@tool
def rag_tool(query):
    """Retrieve relevant information from the pdf doucment.
    use this tool when the user ask factual /conceptual question that might be answered from the stored pdf document. 
    """
    result=retriever.invoke(query)
    context=[doc.page_content for doc in result]
    metadata=[doc.metadata for doc in result]
    return {
        "query":query,
        "context":context,
        "metadata":metadata}

In [25]:
tools =[rag_tool]
llm_with_tools=llm.bind_tools(tools)


In [26]:
class Chat(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [27]:
def chat_node(chat:Chat):
    """A node that takes a list of messages and returns a list of messages."""
    messages=chat["messages"]
    response=llm_with_tools.invoke(messages)
    return {"messages":[response]}

In [30]:
tool_node=ToolNode(tools)

In [34]:
graph=StateGraph(Chat)
graph.add_node('chat_node',chat_node)
graph.add_node('tools',tool_node)

graph.add_edge(START,'chat_node')
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')

chatbot=graph.compile()

In [51]:
answer=chatbot.invoke({"messages":[SystemMessage(content="You are a helpful assistant that answers questions based on the provided document."),HumanMessage(content="based on the resume of candidate rate from 1 to 10 that is this a good fit to hire for an ml internship or job  ?")]})

"Based on the resume provided, a rating of 8 suggests that the candidate has a strong foundation in computer science and relevant coursework in machine learning. The internship experience at Internpe (Virtual Internship Program) is also a plus, as it indicates that the candidate has hands-on experience with AI/ML projects.\n\nHowever, to determine if this is a good fit for an ML internship or job, we need to consider other factors such as:\n\n* Relevant skills and tools: The resume mentions relevant coursework in machine learning, data structures & algorithms, and database management. However, it would be beneficial to see specific skills and tools mentioned, such as programming languages, libraries, or frameworks.\n* Projects and achievements: While the internship experience is a plus, it would be great to see specific projects and achievements that demonstrate the candidate's skills and accomplishments in AI/ML.\n* Education and certifications: The candidate has completed a B.Tech pr